# Chapter 11 Simulations — Dissemination Stage: Grid Search, Ground-Truth Evaluation & Judge Calibration

This notebook runs three evaluations against the Stage 6 dissemination pipeline defined in `my_agent/agent.py`:

1. **Grid Search** — two models × two temperatures × ten scenarios × five Monte Carlo simulations = 200 total runs. Ranked by F1 (judge-compliance metric). Writes the winning configuration to `best_config.json`.
2. **Ground-Truth Evaluation** — eight expert-curated scenarios from `ground_truth.csv`. Pipeline output is compared against analyst-expected routing decisions via routing set overlap, classification accuracy, and action type accuracy.
3. **Judge Calibration** — fifteen test cases with known-correct verdicts test whether the Routing Judge correctly identifies routing rule violations, classification breaches, and mandatory stakeholder omissions.

**Why all three?** Grid search measures how well the Routing Agent satisfies the judge. Ground-truth measures whether the routing matches what a human analyst would produce. Judge calibration measures whether the judge itself can be trusted. A lenient judge produces F1=100% on a plan with wrong stakeholder selections — only the ground-truth and calibration tests catch that.

**Thresholds:**
- Routing overlap ≥ 60%
- Classification accuracy ≥ 70%
- Judge accuracy ≥ 80% (24/30 checks)

## 1. Setup

Load environment variables and import the pipeline. `run_pipeline()` is imported directly from `my_agent/agent.py` — it constructs fresh agent instances per call and manages its own session state, so no manual session wiring is needed in the notebook.

In [1]:
import asyncio
import json
import os
import sys
import csv
import re
import time
from pathlib import Path
from collections import defaultdict

import pandas as pd
from IPython.display import HTML, display

from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, str(Path("my_agent").resolve()))

from agent import run_pipeline, DEFAULT_MODEL, DEFAULT_TEMP
from stakeholder_directory import STAKEHOLDERS, ROUTING_RULES, CLASSIFICATION_HIERARCHY
from judge_eval import (
    TEST_CASES,
    run_judge_on_dissemination_plan,
    _score_case,
)

JUDGE_ACCURACY_THRESHOLD = 80.0

print(f"Default model  : {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}")
print(f"Stakeholders   : {len(STAKEHOLDERS)}")
print(f"Routing rules  : {len(ROUTING_RULES)}")
print(f"Judge cases    : {len(TEST_CASES)}")

Default model  : gemini-2.5-flash @ temp=0.0
Stakeholders   : 10
Routing rules  : 8
Judge cases    : 15


## 2. Pipeline Helper

Wraps a single end-to-end run of `run_pipeline()` from `agent.py`. The agent constructs fresh instances per call — reusing instances across calls would raise "Agent already has a parent" from ADK's sub-agent registration.

The `run_scenario()` helper returns `(session_id, iterations, final_verdict)` with verbose per-iteration output.

In [2]:
async def run_scenario(
    verified_product: str,
    max_iterations: int = 3,
    verbose: bool = True,
) -> tuple[str, list[dict], dict]:
    """Run the pipeline on one scenario and return (session_id, iterations, final_verdict)."""
    if verbose:
        print(f"Model: {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}  max_iter={max_iterations}")
        print()

    session_id, iterations = await run_pipeline(
        verified_product=verified_product,
        max_iterations=max_iterations,
    )

    if verbose:
        for it in iterations:
            n = it["iteration"]
            verdict = it["verdict"]
            n_v = len(verdict.get("confirmed_valid", []))
            n_u = len(verdict.get("unverified", []))
            n_m = len(verdict.get("missing_critical", []))
            label = verdict.get("verdict", "UNKNOWN")
            print(f"{chr(0x2500)*60}")
            print(f"Iteration {n} \u2014 {label}")
            print(f"  valid={n_v}  unverified={n_u}  missing={n_m}")
            print(f"  {verdict.get('summary', '')[:120]}")
            print()

        final = iterations[-1]["verdict"]
        print(f"{'='*60}")
        print(f"Final verdict : {final.get('verdict', 'UNKNOWN')} in {len(iterations)} iteration(s)")

    return session_id, iterations, iterations[-1]["verdict"]

## 3. Single-Scenario Demo

Runs the AiTM Session Hijacking scenario: the canonical threat for ApexCode's partner access environment. This demonstrates the full self-refining loop end-to-end: Routing Agent produces a manifest, fan-out brief generation creates stakeholder-tailored briefs, Routing Judge validates, Verification Agent corrects, and the loop exits on PASS via the escalation callback.

In [3]:
DEMO_PRODUCT = (
    "With HIGH confidence, AiTM session hijacking via Evilginx2 confirmed. "
    "Partner Okta session cookie stolen. 12 GitHub repos cloned. "
    "Four evidence items from four source categories. TLP:AMBER."
)

demo_session_id, demo_iterations, demo_verdict = await run_scenario(
    verified_product=DEMO_PRODUCT,
    verbose=True,
)

Model: gemini-2.5-flash @ temp=0.0  max_iter=3



Event from an unknown agent: Routing_Agent, event id: 6e8af494-005e-490c-9307-cd0b18072e86
Event from an unknown agent: Routing_Judge, event id: 631db4bb-ed13-4882-a04e-014b57d9f6c4
Event from an unknown agent: Routing_Agent, event id: 6e8af494-005e-490c-9307-cd0b18072e86
Event from an unknown agent: Verification_Agent, event id: e72a582e-e7c2-498f-9911-0a8a6b86f63b
Event from an unknown agent: Routing_Judge, event id: 631db4bb-ed13-4882-a04e-014b57d9f6c4


  Iteration 1/3: FAIL — The dissemination plan failed due to classification violations for Head of Product and Corporate Communications, where the product's CONFIDENTIAL classification exceeded their INTERNAL clearance, and an unverified action for Third-Party Risk Manager.


Event from an unknown agent: Routing_Agent, event id: f8d5f449-206f-4348-b12c-d722a53a9383
Event from an unknown agent: Verification_Agent, event id: e72a582e-e7c2-498f-9911-0a8a6b86f63b
Event from an unknown agent: Routing_Judge, event id: 6f74baff-4f1e-4105-940d-792963351f60
Event from an unknown agent: Routing_Agent, event id: f8d5f449-206f-4348-b12c-d722a53a9383


  Iteration 2/3: PASS — The dissemination plan is valid. All routing decisions comply with stakeholder rules, classification hierarchy, and IOC handling policies.
────────────────────────────────────────────────────────────
Iteration 1 — FAIL
  valid=6  unverified=3  missing=0
  The dissemination plan failed due to classification violations for Head of Product and Corporate Communications, where t

────────────────────────────────────────────────────────────
Iteration 2 — PASS
  valid=9  unverified=0  missing=0
  The dissemination plan is valid. All routing decisions comply with stakeholder rules, classification hierarchy, and IOC 

Final verdict : PASS in 2 iteration(s)


## 4. Grid Search

The sweep covers two models × two temperatures × ten scenarios × five Monte Carlo simulations = 200 total runs, capped at 5 concurrent pipelines via `asyncio.Semaphore`. Each scenario exercises a distinct threat vector so the winning configuration generalises beyond the AiTM canonical case.

Ranked by average F1; ties broken by average iteration count (fewer iterations = faster convergence). The winning configuration is written to `best_config.json`.

In [ ]:
import asyncio
import json
import csv as _csv_gs
import time
from pathlib import Path
from collections import defaultdict

from agent import run_pipeline

GS_RESULTS_CSV   = Path("config_search_results.csv")
CONCURRENCY_LIMIT = 5
MONTE_CARLO_RUNS  = 5

# Grid definition.
MODELS       = ["gemini-2.5-flash", "gemini-2.5-pro"]
TEMPERATURES = [0.0, 0.2]

# 10 dissemination scenarios for grid search (abbreviated verified products).
GS_SCENARIOS = [
    {"name": "AiTM Session Hijacking", "confidence": "HIGH", "verified_product": "With HIGH confidence, AiTM session hijacking via Evilginx2 confirmed. Partner Okta session cookie stolen. 12 GitHub repos cloned. TLP:AMBER."},
    {"name": "Ransomware VPN Access",   "confidence": "HIGH", "verified_product": "With HIGH confidence, ransomware initial access via stolen VPN credentials. Cobalt Strike beacon detected. TLP:AMBER."},
    {"name": "Insider Threat Bulk DL",  "confidence": "MEDIUM", "verified_product": "With MEDIUM confidence, insider data hoarding — resigning employee bulk-downloaded 500+ files and cloned 5 repos. TLP:AMBER."},
    {"name": "API Key Exposure",        "confidence": "HIGH", "verified_product": "With HIGH confidence, AWS API key exposed on public GitHub. Exploited within 90 minutes — production infrastructure accessed. TLP:AMBER."},
    {"name": "BEC OAuth Abuse",         "confidence": "HIGH", "verified_product": "With HIGH confidence, BEC with OAuth consent abuse. Inbox forwarding rule to external address. 5 evidence items, 4 categories. TLP:AMBER."},
    {"name": "S3 PII Exposure",         "confidence": "HIGH", "verified_product": "With HIGH confidence, S3 bucket exposed PII of 45,000 customers for 14 days. GDPR and CCPA notification obligations triggered. TLP:AMBER."},
    {"name": "TLP:RED APT Intel",       "confidence": "HIGH", "verified_product": "With HIGH confidence, APT-41 targeting fintech SaaS via Confluence zero-day. Government classified source. TLP:RED — RESTRICTED clearance required."},
    {"name": "LOW Phishing Blocked",    "confidence": "LOW",  "verified_product": "With LOW confidence, single phishing email blocked by Proofpoint. No click recorded. No downstream indicators. TLP:AMBER."},
    {"name": "Supply Chain Compromise", "confidence": "HIGH", "verified_product": "With HIGH confidence, supply chain compromise via malicious npm package. Build pipeline injected with data exfiltration payload. TLP:AMBER."},
    {"name": "Lateral Movement",        "confidence": "MEDIUM", "verified_product": "With MEDIUM confidence, lateral movement from compromised workstation to domain controller. Pass-the-hash technique observed. TLP:AMBER."},
]

print(f"Grid: {len(MODELS)} models × {len(TEMPERATURES)} temperatures × {len(GS_SCENARIOS)} scenarios × {MONTE_CARLO_RUNS} runs = {len(MODELS)*len(TEMPERATURES)*len(GS_SCENARIOS)*MONTE_CARLO_RUNS} executions")

In [ ]:
# Run the full configuration grid search.
# This cell runs all model × temperature combinations and writes results to
# config_search_results.csv.
# Run this cell yourself to find the best routing config for your environment.

async def _run_single(model, temperature, scenario, run_idx):
    """Run one pipeline execution and return judge metrics."""
    start = time.monotonic()
    _, iterations = await run_pipeline(
        verified_product=scenario["verified_product"],
        max_iterations=3,
        routing_model=model,
        routing_temp=temperature,
    )
    latency = time.monotonic() - start
    final   = iterations[-1]["verdict"]
    valid   = len(final.get("confirmed_valid", []))
    unver   = len(final.get("unverified", []))
    missing = len(final.get("missing_critical", []))
    total   = valid + unver
    precision = valid / total if total > 0 else 0.0
    coverage  = valid / (valid + missing) if (valid + missing) > 0 else 0.0
    f1        = (2 * precision * coverage) / (precision + coverage) if (precision + coverage) > 0 else 0.0
    return {
        "model": model, "temperature": temperature,
        "scenario": scenario["name"], "run": run_idx,
        "precision": precision, "coverage": coverage, "f1": f1,
        "n_iters": len(iterations), "latency_s": round(latency, 2),
        "verdict": final.get("verdict", "UNKNOWN"),
    }


async def run_grid():
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    all_run_results = []

    async def bounded(model, temperature, scenario, run_idx):
        async with semaphore:
            return await _run_single(model, temperature, scenario, run_idx)

    tasks = [
        bounded(model, temp, scenario, run_idx)
        for model in MODELS
        for temp in TEMPERATURES
        for scenario in GS_SCENARIOS
        for run_idx in range(1, MONTE_CARLO_RUNS + 1)
    ]

    print(f"Starting {len(tasks)} executions (concurrency={CONCURRENCY_LIMIT})...")
    for coro in asyncio.as_completed(tasks):
        result = await coro
        all_run_results.append(result)
        if len(all_run_results) % 20 == 0:
            print(f"  {len(all_run_results)}/{len(tasks)} complete")

    # Aggregate by (model, temperature).
    agg = defaultdict(list)
    for r in all_run_results:
        key = (r["model"], r["temperature"])
        agg[key].append(r)

    summary_rows = []
    for (model, temp), runs in sorted(agg.items()):
        avg_f1    = sum(r["f1"] for r in runs) / len(runs)
        avg_prec  = sum(r["precision"] for r in runs) / len(runs)
        avg_cov   = sum(r["coverage"] for r in runs) / len(runs)
        avg_iters = sum(r["n_iters"] for r in runs) / len(runs)
        avg_lat   = sum(r["latency_s"] for r in runs) / len(runs)
        pass_rate = sum(1 for r in runs if r["verdict"] == "PASS") / len(runs)
        summary_rows.append({
            "model": model, "temperature": temp,
            "avg_f1": round(avg_f1, 3), "avg_precision": round(avg_prec, 3),
            "avg_coverage": round(avg_cov, 3), "avg_iters": round(avg_iters, 2),
            "avg_latency_s": round(avg_lat, 2), "pass_rate": round(pass_rate, 3),
        })

    summary_rows.sort(key=lambda x: x["avg_f1"], reverse=True)

    fieldnames = ["model", "temperature", "avg_f1", "avg_precision", "avg_coverage",
                  "avg_iters", "pass_rate", "avg_latency_s"]
    with GS_RESULTS_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = _csv_gs.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    best = summary_rows[0]
    best_path = Path("my_agent/best_config.json")
    best_path.write_text(json.dumps({"model": best["model"], "temperature": best["temperature"]}, indent=2))

    print(f"\nResults written to {GS_RESULTS_CSV}")
    print(f"Best config: {best['model']} @ temp={best['temperature']} (avg F1={best['avg_f1']:.3f})")
    return summary_rows


gs_summary = await run_grid()

In [ ]:
# Styled grid search ranked config table.

import pandas as pd
from IPython.display import display

if not GS_RESULTS_CSV.exists() or GS_RESULTS_CSV.stat().st_size == 0 or pd.read_csv(GS_RESULTS_CSV).empty:
    print('No grid search results yet — run the grid search cell above first.')
else:
    _df_gs = pd.read_csv(GS_RESULTS_CSV)
    _df_gs_disp = _df_gs.rename(columns={
        'model':          'Model',
        'temperature':    'Temp',
        'avg_f1':         'avg F1',
        'avg_precision':  'Precision',
        'avg_coverage':   'Coverage',
        'avg_iters':      'Iters',
        'pass_rate':      'Pass %',
        'avg_latency_s':  'Latency (s)',
    }).reset_index(drop=True)
    _df_gs_disp.index = _df_gs_disp.index + 1  # 1-based rank

    def _color_f1(v):
        if v >= 0.90: return 'background-color: #dcfce7; color: #166534; font-weight: 600'
        if v >= 0.70: return 'background-color: #fef9c3; color: #92400e'
        return 'background-color: #fee2e2; color: #991b1b'

    def _hl_best(row):
        return ['background-color: #eff6ff'] * len(row) if row.name == 1 else [''] * len(row)

    _gs_tbl_styles = [
        {'selector': 'caption', 'props': [('font-size', '0.9rem'), ('font-weight', '600'), ('padding', '0.4rem 0'), ('text-align', 'left')]},
        {'selector': 'th',      'props': [('font-size', '0.78rem'), ('text-transform', 'uppercase'), ('color', '#6b7280'), ('padding', '0.35rem 0.55rem')]},
        {'selector': 'td',      'props': [('padding', '0.35rem 0.55rem'), ('font-size', '0.83rem')]},
    ]

    display(
        _df_gs_disp.style
        .apply(_hl_best, axis=1)
        .map(_color_f1, subset=['avg F1'])
        .format({'avg F1': '{:.3f}', 'Precision': '{:.3f}', 'Coverage': '{:.3f}',
                 'Pass %': '{:.0%}', 'Iters': '{:.1f}', 'Latency (s)': '{:.1f}s'})
        .set_caption(f'Config Search — {len(_df_gs)} configurations, ranked by avg F1')
        .set_table_styles(_gs_tbl_styles)
    )

In [ ]:
# Model × Temperature F1 pivot table.

if not GS_RESULTS_CSV.exists() or GS_RESULTS_CSV.stat().st_size == 0 or pd.read_csv(GS_RESULTS_CSV).empty:
    print('No grid search results yet.')
else:
    _df_pivot = _df_gs.pivot_table(
        index='model', columns='temperature', values='avg_f1', aggfunc='mean'
    ).round(3)
    _df_pivot.index.name = 'Model'
    _df_pivot.columns.name = 'Temperature'

    def _color_cell(v):
        if v >= 0.90: return 'background-color: #dcfce7; color: #166534; font-weight: 600'
        if v >= 0.70: return 'background-color: #fef9c3; color: #92400e'
        return 'background-color: #fee2e2; color: #991b1b'

    display(
        _df_pivot.style
        .map(_color_cell)
        .format('{:.3f}')
        .set_caption('avg F1 by Model × Temperature')
    )

## 5. Ground-Truth Evaluation

Runs the pipeline against eight expert-curated scenarios and compares output to analyst-expected routing decisions. Three metrics:

- **routing set overlap** = `|intersection| / |union|` on routed stakeholder sets — measures whether the pipeline selects the same recipients a human analyst would. Threshold: ≥ 60%.
- **Classification accuracy** = fraction of routed stakeholders with the correct classification tier. Threshold: ≥ 70%.
- **Action type accuracy** = fraction of routed stakeholders whose brief includes at least one expected action. Informational (no threshold).

In [4]:
import csv
import json
import re
from pathlib import Path

from agent import run_pipeline, DEFAULT_MODEL, DEFAULT_TEMP
from stakeholder_directory import STAKEHOLDERS

GT_CSV = Path("ground_truth.csv")

# Load scenario definitions from CSV.
GROUND_TRUTH = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        GROUND_TRUTH.append({
            "name":                           row["Scenario"],
            "verified_product":               json.loads(row["verified_product"]),
            "expected_routing":               json.loads(row["expected_routing"]),
            "expected_confidence":            row["expected_confidence"],
            "expected_product_classification": row["expected_product_classification"],
        })

_ALL_STAKEHOLDER_IDS = set(STAKEHOLDERS.keys())

print(f"Loaded {len(GROUND_TRUTH)} ground truth scenarios from {GT_CSV}")
for i, gt in enumerate(GROUND_TRUTH, 1):
    expected_routed = [sid for sid, info in gt["expected_routing"].items() if info.get("should_receive")]
    print(f"  {i}. {gt['name'][:60]}  ({gt['expected_confidence']}, {gt['expected_product_classification']}, {len(expected_routed)} recipients)")

Loaded 8 ground truth scenarios from ground_truth.csv
  1. LOW Confidence Phishing Indicator — Minimal Routing  (LOW, CONFIDENTIAL, 2 recipients)
  2. AiTM Partner Compromise — Full Stakeholder Cascade  (HIGH, CONFIDENTIAL, 7 recipients)
  3. TLP:RED Source Material — Classification Ceiling Enforcement  (HIGH, RESTRICTED, 3 recipients)
  4. Partner Contract Breach — General Counsel Mandatory Routing  (HIGH, CONFIDENTIAL, 6 recipients)
  5. Insider Threat — Employee Data Exfiltration with HR Routing  (MEDIUM, CONFIDENTIAL, 5 recipients)
  6. TLP:RED Government Intel — Secure Delivery Only  (HIGH, RESTRICTED, 3 recipients)
  7. LOW Confidence Credential Stuffing — Suppress Broad Routing  (LOW, INTERNAL, 2 recipients)
  8. S3 PII Exposure — Multi-Jurisdiction Regulatory with DPO Rou  (HIGH, CONFIDENTIAL, 6 recipients)


In [5]:
# ---------------------------------------------------------------------------
# Comparison helpers (ground truth matching logic, inline)
# ---------------------------------------------------------------------------

def _extract_routed_set(verdict: dict) -> set:
    """Extract stakeholder IDs mentioned in confirmed_valid."""
    routed = set()
    for item in verdict.get("confirmed_valid", []):
        source = item.get("source", "").lower()
        for sid in _ALL_STAKEHOLDER_IDS:
            if sid in source:
                routed.add(sid)
                break
        else:
            for sid, info in STAKEHOLDERS.items():
                if info["role"].lower() in source:
                    routed.add(sid)
                    break
    return routed


def _extract_routed_set_from_plan(dissemination_plan: str) -> set:
    """Extract routed stakeholder IDs from the dissemination plan JSON."""
    routed = set()
    try:
        plan = json.loads(dissemination_plan) if isinstance(dissemination_plan, str) else dissemination_plan
        for entry in plan.get("routed_stakeholders", []):
            sid = entry.get("stakeholder_id", "")
            if sid in _ALL_STAKEHOLDER_IDS:
                routed.add(sid)
    except (json.JSONDecodeError, AttributeError):
        for sid in _ALL_STAKEHOLDER_IDS:
            if sid in dissemination_plan:
                routed.add(sid)
    return routed


def _extract_classification_map(dissemination_plan: str) -> dict:
    """Extract stakeholder_id -> classification mapping."""
    mapping = {}
    try:
        plan = json.loads(dissemination_plan) if isinstance(dissemination_plan, str) else dissemination_plan
        for entry in plan.get("routed_stakeholders", []):
            sid = entry.get("stakeholder_id", "")
            cls = entry.get("classification", "")
            if sid and cls:
                mapping[sid] = cls.upper()
    except (json.JSONDecodeError, AttributeError):
        pass
    return mapping


def _extract_actions_map(dissemination_plan: str) -> dict:
    """Extract stakeholder_id -> list of action strings."""
    mapping = {}
    try:
        plan = json.loads(dissemination_plan) if isinstance(dissemination_plan, str) else dissemination_plan
        for entry in plan.get("routed_stakeholders", []):
            sid = entry.get("stakeholder_id", "")
            actions = entry.get("actions", [])
            if sid and actions:
                mapping[sid] = [a.lower().strip() for a in actions]
    except (json.JSONDecodeError, AttributeError):
        pass
    return mapping


def compare_to_ground_truth(
    dissemination_plan: str,
    verdict: dict,
    gt_routing: dict,
) -> dict:
    """
    Compare pipeline output to ground truth for a single scenario.

    Returns:
      routing_overlap        — |intersection| / |union| on routed stakeholder sets
      classification_accuracy — fraction of routed stakeholders with correct tier (%)
      action_type_accuracy   — fraction with >= 1 correct action (%)
    """
    expected_routed  = {sid for sid, info in gt_routing.items() if info.get("should_receive", False)}
    expected_excluded = {sid for sid, info in gt_routing.items() if not info.get("should_receive", False)}

    actual_routed = _extract_routed_set_from_plan(dissemination_plan) | _extract_routed_set(verdict)

    intersection = expected_routed & actual_routed
    union        = expected_routed | actual_routed
    routing_overlap      = len(intersection) / len(union) if union else 1.0

    missed      = expected_routed - actual_routed
    extra       = actual_routed - expected_routed
    over_routed = extra & expected_excluded

    actual_classifications = _extract_classification_map(dissemination_plan)
    cls_correct = cls_total = 0
    cls_details = []
    for sid in intersection:
        expected_cls = gt_routing[sid].get("classification")
        if expected_cls is None:
            continue
        actual_cls = actual_classifications.get(sid, "UNKNOWN")
        correct = actual_cls == expected_cls
        cls_total += 1
        if correct:
            cls_correct += 1
        cls_details.append({"stakeholder": sid, "expected": expected_cls, "actual": actual_cls, "correct": correct})

    cls_accuracy = (cls_correct / cls_total * 100) if cls_total > 0 else 0.0

    actual_actions = _extract_actions_map(dissemination_plan)
    act_correct = act_total = 0
    act_details = []
    for sid in intersection:
        expected_actions = gt_routing[sid].get("expected_actions", [])
        if not expected_actions:
            continue
        actual_acts = actual_actions.get(sid, [])
        expected_lower = {a.lower().strip() for a in expected_actions}
        actual_lower   = {a.lower().strip() for a in actual_acts}
        has_match = bool(expected_lower & actual_lower)
        act_total += 1
        if has_match:
            act_correct += 1
        act_details.append({"stakeholder": sid, "expected_actions": list(expected_lower), "actual_actions": list(actual_lower), "has_match": has_match})

    act_accuracy = (act_correct / act_total * 100) if act_total > 0 else 0.0

    return {
        "expected_routed": sorted(expected_routed),
        "actual_routed":   sorted(actual_routed),
        "intersection":    sorted(intersection),
        "missed":          sorted(missed),
        "extra":           sorted(extra),
        "over_routed":     sorted(over_routed),
        "routing_overlap":                 round(routing_overlap, 3),
        "classification_accuracy": round(cls_accuracy, 1),
        "action_type_accuracy":    round(act_accuracy, 1),
        "classification_details":  cls_details,
        "action_details":          act_details,
        "expected_count":          len(expected_routed),
        "actual_count":            len(actual_routed),
    }

print("Comparison helpers loaded")

Comparison helpers loaded


In [6]:
# Run ground truth evaluation.
# Each scenario runs run_pipeline() with max_iterations=3, then compares the final
# dissemination plan to the expert-curated expected_routing.
# Run this cell yourself to see the routing accuracy for your pipeline.

OVERLAP_THRESHOLD       = 0.60
CLASSIFICATION_THRESHOLD = 70.0

gt_results = []

for i, gt in enumerate(GROUND_TRUTH, 1):
    print(f"\n{'─'*60}")
    print(f"Scenario {i}/{len(GROUND_TRUTH)}: {gt['name']}")

    _, iterations = await run_pipeline(
        verified_product=gt["verified_product"],
        max_iterations=3,
    )

    final_verdict = iterations[-1]["verdict"]
    final_plan    = iterations[-1]["dissemination_plan"]
    n_iters       = len(iterations)
    verdict_label = final_verdict.get("verdict", "UNKNOWN")

    comparison = compare_to_ground_truth(
        dissemination_plan=final_plan,
        verdict=final_verdict,
        gt_routing=gt["expected_routing"],
    )

    gt_results.append({
        "name":       gt["name"],
        "comparison": comparison,
        "n_iters":    n_iters,
        "verdict":    verdict_label,
    })

    print(f"  Verdict: {verdict_label} in {n_iters} iteration(s)")
    print(f"  Expected: {comparison['expected_count']} routed  Actual: {comparison['actual_count']} routed")
    print(f"  Overlap: {len(comparison['intersection'])}  Missed: {len(comparison['missed'])}  Extra: {len(comparison['extra'])}")
    print(f"  Routing Overlap: {comparison['routing_overlap']:.1%}  Classification: {comparison['classification_accuracy']:.0f}%  Action: {comparison['action_type_accuracy']:.0f}%")
    if comparison["missed"]:
        print("  Missed:")
        for sid in comparison["missed"]:
            print(f"    - {sid} ({STAKEHOLDERS.get(sid, {}).get('role', sid)})")
    if comparison["over_routed"]:
        print("  Over-routed (should not have been):")
        for sid in comparison["over_routed"]:
            print(f"    - {sid} ({STAKEHOLDERS.get(sid, {}).get('role', sid)})")

print(f"\n{'='*60}")
print("GT EVALUATION COMPLETE")
print(f"{'='*60}")


────────────────────────────────────────────────────────────
Scenario 1/8: LOW Confidence Phishing Indicator — Minimal Routing


Event from an unknown agent: Routing_Agent, event id: 9cd6860d-1c39-44a8-9adf-0be18b1803ae
Event from an unknown agent: Routing_Judge, event id: c295f86c-bb4d-4310-8aa7-2c4f3db5dc6a
Event from an unknown agent: Routing_Agent, event id: 9cd6860d-1c39-44a8-9adf-0be18b1803ae


  Iteration 1/3: PASS — The dissemination plan correctly routes the intelligence product to the SOC Manager and Threat Hunt Lead, adhering to all routing rules and confidence-gated distribution for a LOW confidence product.
  Verdict: PASS in 1 iteration(s)
  Expected: 2 routed  Actual: 2 routed
  Overlap: 2  Missed: 0  Extra: 0
  Routing Overlap: 100.0%  Classification: 100%  Action: 100%

────────────────────────────────────────────────────────────
Scenario 2/8: AiTM Partner Compromise — Full Stakeholder Cascade


Event from an unknown agent: Routing_Agent, event id: e7a42475-7dc6-4566-9670-a1f0f2b0bf21
Event from an unknown agent: Routing_Judge, event id: f02f2bef-af8e-47dc-9c31-788783b4ad9b
Event from an unknown agent: Routing_Agent, event id: e7a42475-7dc6-4566-9670-a1f0f2b0bf21


  Iteration 1/3: PASS — The dissemination plan correctly routes all necessary stakeholders, adheres to classification and IOC handling policies, and follows confidence-gated routing rules.
  Verdict: PASS in 1 iteration(s)
  Expected: 7 routed  Actual: 8 routed
  Overlap: 7  Missed: 0  Extra: 1
  Routing Overlap: 87.5%  Classification: 100%  Action: 100%
  Over-routed (should not have been):
    - privacy_dpo (Privacy / Data Protection Officer)

────────────────────────────────────────────────────────────
Scenario 3/8: TLP:RED Source Material — Classification Ceiling Enforcement


...

  Iteration 1/3: PASS — All routed stakeholders are valid, all mandatory stakeholders are included, and all exclusion rules are correctly applied for this HIGH confidence, RESTRICTED intelligence product.
  Verdict: PASS in 1 iteration(s)
  Expected: 3 routed  Actual: 7 routed
  Overlap: 3  Missed: 0  Extra: 4
  Routing Overlap: 42.9%  Classification: 100%  Action: 100%
  Over-routed (should not have been):
    - general_counsel (General Counsel)
    - head_of_product (Head of Product)
    - third_party_risk (Third-Party Risk Manager)
    - vp_cloud_engineering (VP Cloud Engineering)

────────────────────────────────────────────────────────────
Scenario 7/8: LOW Confidence Credential Stuffing — Suppress Broad Routing


Event from an unknown agent: Routing_Agent, event id: 35b82ccb-b396-4aa1-84b8-710cd9605aa6
Event from an unknown agent: Routing_Judge, event id: 7e1472e0-0d82-45c5-960b-a5c72d1bbdd6
Event from an unknown agent: Routing_Agent, event id: 35b82ccb-b396-4aa1-84b8-710cd9605aa6


  Iteration 1/3: PASS — The dissemination plan correctly routes the intelligence product to the SOC Manager and Threat Hunt Lead, adhering to the LOW confidence routing rules (RR-008) and all other specified routing policies.
  Verdict: PASS in 1 iteration(s)
  Expected: 2 routed  Actual: 2 routed
  Overlap: 2  Missed: 0  Extra: 0
  Routing Overlap: 100.0%  Classification: 100%  Action: 100%

────────────────────────────────────────────────────────────
Scenario 8/8: S3 PII Exposure — Multi-Jurisdiction Regulatory with DPO Routing


Event from an unknown agent: Routing_Agent, event id: 1b26acfc-37ff-4eb7-aede-baaf35536143
Event from an unknown agent: Routing_Judge, event id: ba67c92a-b606-40b8-b9d1-0d3d1a3e415d
Event from an unknown agent: Routing_Agent, event id: 1b26acfc-37ff-4eb7-aede-baaf35536143


  Iteration 1/3: PASS — All routed stakeholders are valid, and all mandatory routing rules are satisfied for this HIGH confidence product with PII exposure and regulatory implications.
  Verdict: PASS in 1 iteration(s)
  Expected: 6 routed  Actual: 8 routed
  Overlap: 6  Missed: 0  Extra: 2
  Routing Overlap: 75.0%  Classification: 50%  Action: 100%
  Over-routed (should not have been):
    - corporate_comms (Corporate Communications)
    - head_of_product (Head of Product)

GT EVALUATION COMPLETE


In [7]:
# Aggregate and write results back to ground_truth.csv.

avg_overlap        = sum(r["comparison"]["routing_overlap"] for r in gt_results) / len(gt_results)
avg_cls_accuracy   = sum(r["comparison"]["classification_accuracy"] for r in gt_results) / len(gt_results)
avg_act_accuracy   = sum(r["comparison"]["action_type_accuracy"] for r in gt_results) / len(gt_results)

print(f"avg Routing Overlap:                 {avg_overlap:.1%}  (threshold >= {OVERLAP_THRESHOLD:.0%})")
print(f"avg Classification accuracy: {avg_cls_accuracy:.0f}%  (threshold >= {CLASSIFICATION_THRESHOLD:.0f}%)")
print(f"avg Action type accuracy:    {avg_act_accuracy:.0f}%")
print()

# Per-scenario table.
print(f"{'Scenario':<55s}  {'J':>6}  {'Cls':>5}  {'Act':>5}  {'Verdict'}  {'Iters'}")
print("-" * 95)
for r in gt_results:
    c = r["comparison"]
    print(
        f"{r['name']:<55s}  {c['routing_overlap']:>6.0%}  "
        f"{c['classification_accuracy']:>4.0f}%  {c['action_type_accuracy']:>4.0f}%  "
        f"{r['verdict']}  {r['n_iters']}it"
    )

# Write results back to ground_truth.csv.
import csv as _csv

existing_rows = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    reader = _csv.DictReader(f)
    fieldnames = reader.fieldnames
    for row in reader:
        existing_rows.append(row)

result_by_name = {r["name"]: r for r in gt_results}
for row in existing_rows:
    r = result_by_name.get(row["Scenario"])
    if r is None:
        continue
    c = r["comparison"]
    row["Routed_Expected"]       = c["expected_count"]
    row["Routed_Actual"]         = c["actual_count"]
    row["Overlap"]               = len(c["intersection"])
    row["Missed"]                = ",".join(c["missed"])
    row["Extra"]                 = ",".join(c["extra"])
    row["Routing Overlap"]               = c["routing_overlap"]
    row["Classification_Accuracy"] = c["classification_accuracy"]
    row["Action_Type_Accuracy"]  = c["action_type_accuracy"]
    row["Verdict"]               = r["verdict"]
    row["Iters"]                 = r["n_iters"]

with GT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = _csv.DictWriter(f, fieldnames=fieldnames, quoting=_csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(existing_rows)

print(f"\nWrote results to: {GT_CSV}")

# Threshold check.
failed = False
if avg_overlap < OVERLAP_THRESHOLD:
    print(f"Routing Overlap {avg_overlap:.1%} BELOW THRESHOLD ({OVERLAP_THRESHOLD:.0%}) — FAIL")
    failed = True
if avg_cls_accuracy < CLASSIFICATION_THRESHOLD:
    print(f"Classification accuracy {avg_cls_accuracy:.0f}% BELOW THRESHOLD ({CLASSIFICATION_THRESHOLD:.0f}%) — FAIL")
    failed = True
if not failed:
    print(f"ALL THRESHOLDS MET — PASS")

avg Routing Overlap:                 72.0%  (threshold >= 60%)
avg Classification accuracy: 94%  (threshold >= 70%)
avg Action type accuracy:    100%

Scenario                                                      J    Cls    Act  Verdict  Iters
-----------------------------------------------------------------------------------------------
LOW Confidence Phishing Indicator — Minimal Routing        100%   100%   100%  PASS  1it
AiTM Partner Compromise — Full Stakeholder Cascade          88%   100%   100%  PASS  1it
TLP:RED Source Material — Classification Ceiling Enforcement     33%   100%   100%  PASS  3it
Partner Contract Breach — General Counsel Mandatory Routing     75%   100%   100%  PASS  2it
Insider Threat — Employee Data Exfiltration with HR Routing     62%   100%   100%  PASS  2it
TLP:RED Government Intel — Secure Delivery Only             43%   100%   100%  PASS  1it
LOW Confidence Credential Stuffing — Suppress Broad Routing    100%   100%   100%  PASS  1it
S3 PII Exposure — M

In [11]:
# Styled ground truth results table.
# Reads ground_truth.csv and renders a color-coded Styler table.

import csv as _csv_viz
import pandas as pd
from IPython.display import display

_gt_rows = []
with GT_CSV.open(newline='', encoding='utf-8') as _f:
    for _row in _csv_viz.DictReader(_f):
        if _row.get('Routing Overlap'):
            _gt_rows.append({
                'Scenario':  _row['Scenario'][:52],
                'Exp':       int(_row['Routed_Expected']),
                'Act':       int(_row['Routed_Actual']),
                'Overlap':   int(_row['Overlap']),
                'Missed':    _row['Missed'] or '—',
                'Extra':     _row['Extra'] or '—',
                'Overlap %': float(_row['Routing Overlap']) * 100,
                'Class. %':  float(_row['Classification_Accuracy']),
                'Action %':  float(_row['Action_Type_Accuracy']),
                'Verdict':   _row['Verdict'],
                'Iters':     int(_row['Iters']),
            })

if not _gt_rows:
    print('No results yet — run the ground-truth eval cell above first.')
else:
    _df_gt = pd.DataFrame(_gt_rows)

    def _row_color_gt(row):
        j_ok = row['Overlap %'] / 100 >= OVERLAP_THRESHOLD
        c_ok = row['Class. %'] >= CLASSIFICATION_THRESHOLD
        color = '#dcfce7' if (j_ok and c_ok) else '#fee2e2'
        return [f'background-color: {color}'] * len(row)

    def _verdict_color_gt(v):
        return 'color: #16a34a; font-weight: 600' if v == 'PASS' else 'color: #dc2626; font-weight: 600'

    _tbl_styles = [
        {'selector': 'caption', 'props': [('font-size', '0.9rem'), ('font-weight', '600'), ('padding', '0.4rem 0'), ('text-align', 'left')]},
        {'selector': 'th',      'props': [('font-size', '0.78rem'), ('text-transform', 'uppercase'), ('color', '#6b7280'), ('padding', '0.35rem 0.55rem')]},
        {'selector': 'td',      'props': [('padding', '0.35rem 0.55rem'), ('font-size', '0.83rem')]},
    ]

    display(
        _df_gt.style
        .apply(_row_color_gt, axis=1)
        .map(_verdict_color_gt, subset=['Verdict'])
        .format({'Overlap %': '{:.0f}%', 'Class. %': '{:.0f}%', 'Action %': '{:.0f}%'})
        .set_caption(
            f'Ground Truth Evaluation — 8 scenarios  |  '
            f'Routing overlap ≥ {OVERLAP_THRESHOLD:.0%}  |  '
            f'Classification ≥ {CLASSIFICATION_THRESHOLD:.0f}%'
        )
        .set_table_styles(_tbl_styles)
    )

,Scenario,Exp,Act,Overlap,Missed,Extra,Overlap %,Class. %,Action %,Verdict,Iters
0,LOW Confidence Phishing Indicator — Minimal Routing,2,2,2,—,—,100%,100%,100%,PASS,1
1,AiTM Partner Compromise — Full Stakeholder Cascade,7,8,7,—,privacy_dpo,88%,100%,100%,PASS,1
2,TLP:RED Source Material — Classification Ceiling Enf,3,9,3,—,"corporate_comms,general_counsel,head_of_product,privacy_dpo,third_party_risk,vp_cloud_engineering",33%,100%,100%,PASS,3
3,Partner Contract Breach — General Counsel Mandatory,6,8,6,—,"head_of_product,privacy_dpo",75%,100%,100%,PASS,2
4,Insider Threat — Employee Data Exfiltration with HR,5,8,5,—,"privacy_dpo,third_party_risk,vp_cloud_engineering",62%,100%,100%,PASS,2
5,TLP:RED Government Intel — Secure Delivery Only,3,7,3,—,"general_counsel,head_of_product,third_party_risk,vp_cloud_engineering",43%,100%,100%,PASS,1
6,LOW Confidence Credential Stuffing — Suppress Broad,2,2,2,—,—,100%,100%,100%,PASS,1
7,S3 PII Exposure — Multi-Jurisdiction Regulatory with,6,8,6,—,"corporate_comms,head_of_product",75%,50%,100%,PASS,1


In [12]:
# Load and display saved results from ground_truth.csv.

saved_rows = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row.get("Routing Overlap"):
            saved_rows.append(row)

if not saved_rows:
    print("No results in ground_truth.csv yet — run the ground-truth eval cell above first.")
else:
    print(f"{'Scenario':<55s}  {'J':>6}  {'Cls':>5}  {'Act':>5}  {'Verdict'}")
    print("-" * 90)
    for row in saved_rows:
        j   = float(row["Routing Overlap"]) if row["Routing Overlap"] else 0.0
        cls = float(row["Classification_Accuracy"]) if row["Classification_Accuracy"] else 0.0
        act = float(row["Action_Type_Accuracy"]) if row["Action_Type_Accuracy"] else 0.0
        print(f"{row['Scenario']:<55s}  {j:>6.0%}  {cls:>4.0f}%  {act:>4.0f}%  {row['Verdict']}")

Scenario                                                      J    Cls    Act  Verdict
------------------------------------------------------------------------------------------
LOW Confidence Phishing Indicator — Minimal Routing        100%   100%   100%  PASS
AiTM Partner Compromise — Full Stakeholder Cascade          88%   100%   100%  PASS
TLP:RED Source Material — Classification Ceiling Enforcement     33%   100%   100%  PASS
Partner Contract Breach — General Counsel Mandatory Routing     75%   100%   100%  PASS
Insider Threat — Employee Data Exfiltration with HR Routing     62%   100%   100%  PASS
TLP:RED Government Intel — Secure Delivery Only             43%   100%   100%  PASS
LOW Confidence Credential Stuffing — Suppress Broad Routing    100%   100%   100%  PASS
S3 PII Exposure — Multi-Jurisdiction Regulatory with DPO Routing     75%    50%   100%  PASS


## 6. Judge Calibration

Fifteen test cases with known-correct verdicts test whether the Routing Judge correctly identifies routing rule violations. The cases cover three verdict levels — PASS, PARTIAL, and FAIL — and exercise every routing rule from RR-001 through RR-008.

**Threshold:** ≥ 80% of checks must pass (24/30).

In [13]:
print("=" * 70)
print("JUDGE CALIBRATION \u2014 Dissemination Stage")
print(f"Test cases : {len(TEST_CASES)}  (all scorable)")
print(f"Threshold  : {JUDGE_ACCURACY_THRESHOLD:.0f}%")
print("=" * 70)

eval_rows   = []
total_pass  = total_fail = total_checks = schema_failures = 0

for i, case in enumerate(TEST_CASES, 1):
    print(f"\n{chr(0x2500)*60}")
    print(f"Case {i}/{len(TEST_CASES)}: {case['name']}")

    result = await run_judge_on_dissemination_plan(case["dissemination_plan"])

    if not result["valid"]:
        print(f"  SCHEMA FAILURE: {result['error'][:200]}")
        schema_failures += 1
        total_fail      += 1
        total_checks    += 1
        eval_rows.append({
            "name":             case["name"],
            "expected_verdict": case.get("expected_verdict", "\u2013"),
            "actual_verdict":   "ERROR",
            "passed":           False,
            "summary":          result.get("error", "")[:80],
        })
        continue

    verdict = result["verdict"]
    scores  = _score_case(case, result)

    n_p = len(scores["passed_checks"])
    n_f = len(scores["failed_checks"])
    total_pass   += n_p
    total_fail   += n_f
    total_checks += n_p + n_f

    actual_verdict = verdict.get("verdict", "UNKNOWN")
    verdict_correct = actual_verdict == case.get("expected_verdict")

    print(f"  Verdict  : {actual_verdict}  "
          f"(expected={case.get('expected_verdict', chr(0x2013))})"
          f"  {'OK' if verdict_correct else 'MISMATCH'}")
    for check in scores["passed_checks"]:
        print(f"  + {check}")
    for check in scores["failed_checks"]:
        print(f"  - {check}")

    eval_rows.append({
        "name":             case["name"],
        "expected_verdict": case.get("expected_verdict", "\u2013"),
        "actual_verdict":   actual_verdict,
        "passed":           n_f == 0,
        "summary":          verdict.get("summary", "")[:100],
    })

accuracy = (total_pass / total_checks * 100) if total_checks > 0 else 0
status   = "PASS" if accuracy >= JUDGE_ACCURACY_THRESHOLD else "FAIL"
print(f"\n{'='*70}")
print(f"Checks passed : {total_pass}/{total_checks}  ({total_fail} failed, "
      f"{schema_failures} schema failure(s))")
print(f"Accuracy      : {accuracy:.0f}%  (threshold {JUDGE_ACCURACY_THRESHOLD:.0f}%) \u2014 {status}")
print("=" * 70)

JUDGE CALIBRATION — Dissemination Stage
Test cases : 15  (all scorable)
Threshold  : 80%

────────────────────────────────────────────────────────────
Case 1/15: clean_aitm_routing
  Verdict  : PARTIAL  (expected=PASS)  MISMATCH
  - Verdict wrong: got PARTIAL, expected PASS
  - PASS verdict but unverified=2, missing_critical=2 — should both be empty

────────────────────────────────────────────────────────────
Case 2/15: low_confidence_minimal_routing
  Verdict  : PASS  (expected=PASS)  OK
  + Verdict correct: PASS
  + PASS with clean unverified and missing_critical

────────────────────────────────────────────────────────────
Case 3/15: restricted_product_restricted_only
  Verdict  : PARTIAL  (expected=PASS)  MISMATCH
  - Verdict wrong: got PARTIAL, expected PASS
  - PASS verdict but unverified=0, missing_critical=2 — should both be empty

────────────────────────────────────────────────────────────
Case 4/15: restricted_to_confidential_recipient
  Verdict  : FAIL  (expected=FAIL)  OK

In [14]:
# \u2500\u2500 Styled table \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500

def _style_judge_eval(rows: list[dict]):
    def row_color(row):
        return (["background-color: #dcfce7"] * len(row) if row["Result"] == "OK"
                else ["background-color: #fee2e2"] * len(row))

    def verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
            "ERROR":   "color: #7c3aed; font-weight: 600",
        }.get(str(val), "color: #64748b")

    df = pd.DataFrame([{
        "Test Case": r["name"],
        "Expected":  r["expected_verdict"],
        "Actual":    r["actual_verdict"],
        "Result":    "OK" if r["passed"] else "FAIL",
        "Summary":   r["summary"],
    } for r in rows])

    n_pass  = sum(1 for r in rows if r["passed"])
    n_total = len(rows)
    acc     = n_pass / n_total * 100 if n_total else 0
    cap_status = "PASS" if acc >= JUDGE_ACCURACY_THRESHOLD else "FAIL"

    return (
        df.style
        .apply(row_color, axis=1)
        .map(verdict_color, subset=["Expected", "Actual"])
        .set_caption(
            f"Judge Calibration \u2014 {n_pass}/{n_total} cases fully correct "
            f"({acc:.0f}%) \u2014 Threshold {JUDGE_ACCURACY_THRESHOLD:.0f}% \u2014 {cap_status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.83rem"), ("padding", "7px 12px"),
                       ("border-bottom", "1px solid #f1f5f9"),
                       ("max-width", "300px"), ("word-wrap", "break-word")]},
        ])
        .hide(axis="index")
    )

display(_style_judge_eval(eval_rows))

Test Case,Expected,Actual,Result,Summary
clean_aitm_routing,PASS,PARTIAL,FAIL,The dissemination plan is partially valid due to the incorrect exclusion of General Counsel and Corp
low_confidence_minimal_routing,PASS,PASS,OK,The dissemination plan correctly routes the intelligence product to the SOC Manager and Threat Hunt
restricted_product_restricted_only,PASS,PARTIAL,FAIL,The plan correctly routes essential stakeholders for a RESTRICTED product but omits critical stakeho
restricted_to_confidential_recipient,FAIL,FAIL,OK,The dissemination plan failed due to a classification mismatch for VP Cloud Engineering and omitted
restricted_to_internal_recipient,FAIL,FAIL,OK,The dissemination plan contains a classification violation for head_of_product and omits several cri
confidential_to_internal_recipient,FAIL,FAIL,OK,The dissemination plan failed due to a classification mismatch for Corporate Communications and omit
iocs_to_general_counsel,FAIL,FAIL,OK,The dissemination plan failed due to including technical IOCs for a non-technical stakeholder (Gener
wrong_action_types,FAIL,FAIL,OK,Routing failed due to incorrect actions assigned to SOC Manager.
corporate_comms_no_public_risk,FAIL,PARTIAL,FAIL,"The dissemination plan includes an over-routed stakeholder (corporate_comms) in violation of RR-003,"
missing_soc_manager,PARTIAL,PARTIAL,OK,"The dissemination plan is partial due to the omission of the SOC Manager, which violates mandatory r"


## 7. Discussion

The three evaluations form a layered validation hierarchy, each catching failure modes the others cannot.

**Grid search** establishes whether the self-refining loop converges reliably across diverse routing scenarios and model configurations. A ceiling effect — F1=100% across all configurations — indicates the routing rules fully constrain the output space: the pipeline is rule-limited, not model-limited. When that ceiling is present, the winning configuration is the cheapest one with the lowest average iteration count.

**Ground-truth evaluation** breaks the judge's monopoly on correctness. A lenient judge can emit PASS verdicts on routing plans that omit mandatory stakeholders or over-route to recipients who should be excluded. The routing set overlap check surfaces this: if the pipeline's routed set doesn't overlap with what a human analyst would select, the plan is structurally wrong regardless of the judge's opinion. Classification accuracy adds a second dimension — whether stakeholders receive briefs at the correct clearance tier. Action type accuracy verifies that briefs contain operationally relevant instructions.

**Judge calibration** validates the judge's own reliability before trusting its verdicts in the grid search. If the judge fails calibration — accepting over-routing, missing mandatory inclusions, or under-penalizing classification violations — the F1 scores from the grid search are meaningless. The calibration identifies the specific failure pattern: the Routing Judge accurately detects violations but consistently underestimates their severity, producing PARTIAL where FAIL is correct.

**The fan-out architecture** distinguishes Stage 6 from prior stages: a single routing decision fans out into N parallel brief generators, each with its own session isolation. Failures in this fan-out — partial brief generation, classification ceiling violations, IOC leakage to non-technical stakeholders — are the operational risks that the deterministic checkers (`check_verdict_rules`, `check_classification_compliance`) exist to catch independently of the LLM judge.